In [ ]:
#Creating an ML model to approximate Multi-Commodity Flow Network Flow Optimizaiton
#A series of 100,000 samples of data, will split into training and test data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import torch
print(torch.__version__,torch.version.cuda)

!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html

2.11.0+cu130 13.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 75.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 107.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 131.7 MB/s eta 0:00:00


In [ ]:
%%writefile static_network.py
from pathlib import Path
import h5py
import numpy as np

#Establishes an ordering for nodes, edges, and commodities to be used down stream
class StaticNetwork:
    def __init__(self, path: Path):
        path = Path(path)
        with h5py.File(path, "r") as f:
            self.city_order = [c.decode("utf-8") if isinstance(c, bytes) else c
                                for c in f["city_order"][:]]
            self.commodity_order = [c.decode("utf-8") if isinstance(c, bytes) else c
                                     for c in f["commodity_order"][:]]
            self.edge_i = f["edge_i"][:]
            self.edge_j = f["edge_j"][:]
            self.edge_distance = f["edge_distance"][:]
            self.num_cities = int(f.attrs["num_cities"])
            self.num_commodities = int(f.attrs["num_commodities"])
            self.num_edges = int(f.attrs["num_edges"])

        self.city_index = {city: idx for idx, city in enumerate(self.city_order)}
        self.commodity_index = {c: idx for idx, c in enumerate(self.commodity_order)}

    @property
    def N(self) -> int:
        return self.num_cities

    @property
    def K(self) -> int:
        return self.num_commodities



Writing static_network.py


In [ ]:
from static_network import StaticNetwork

static = StaticNetwork("/content/drive/MyDrive/training_data/static_network.h5")
print(static.N, static.K, static.num_edges)

110 5 240


In [ ]:
# graph_dataset.py
import h5py
import numpy as np
import torch
from torch_geometric.data import Data, Dataset

from static_network import StaticNetwork

#pulls in the data and establishes a fixed order for every sample
class GraphTransportationDataset(Dataset):
    def __init__(self, shard_dir, static_network_path):
        super().__init__()
        from pathlib import Path
        self.shard_dir = Path(shard_dir)
        self._open_files = {}  # cache: shard_path -> open h5py.File handle
        #loads the fixed values from the previous class
        self.static = StaticNetwork(static_network_path)

        self.N = self.static.N
        self.K = self.static.K
        self.E = self.static.num_edges

        # Fixed edge_index, built once from the static file — this ordering
        # is what every sample's data gets aligned to.
        self.edge_index = torch.tensor(
            np.stack([self.static.edge_i, self.static.edge_j]), dtype=torch.long
        )

        #Extend edge_index to the virtual node
        real_to_virtual = torch.stack([
            torch.arange(self.N),
            torch.full((self.N,), self.N)
        ])

        virtual_to_real = torch.stack([
            torch.full((self.N,), self.N),
            torch.arange(self.N)
        ])

        #Adding it together

        self.edge_index = torch.cat(
            [self.edge_index,real_to_virtual,virtual_to_real],
            dim=1,
        )

        # (i, j) -> position in the fixed edge list (excludes virtual nodes)
        self.edge_pos = {
            (int(i), int(j)): pos
            for pos, (i, j) in enumerate(zip(self.static.edge_i, self.static.edge_j))
          }

        self.edge_id = torch.cat([
            torch.arange(self.E),
            torch.full((self.N,), self.E),
            torch.full((self.N,), self.E + 1),
            ])

        shard_paths = sorted(self.shard_dir.glob("shard_*.h5"))
        self._index_map = []
        for p in shard_paths:
            with h5py.File(p, "r") as f:
                n = int(f.attrs["num_simulations"])
            for local_idx in range(n):
                self._index_map.append((p, local_idx))



    def len(self):
        return len(self._index_map)

    def get(self, idx):
        shard_path, local_idx = self._index_map[idx]

        if shard_path not in self._open_files:
          self._open_files[shard_path] = h5py.File(shard_path, "r")
        f = self._open_files[shard_path]

        g = f[f"simulations/simulation_{local_idx:06d}"]

        cap_i, cap_j, cap_v = g["capacity/i"][:], g["capacity/j"][:], g["capacity/v"][:]
        #The optimal solution from the sample is loaded, where
        #k is the commodity index
              #i the source node index
              #j is the destination node index
              #v is the actual flow value
        flow_k, flow_i, flow_j, flow_v = (g["optimal_flow/k"][:], g["optimal_flow/i"][:],
                  g["optimal_flow/j"][:], g["optimal_flow/v"][:],)
        supply, demand, price = g["supply"][:], g["demand"][:], g["price"][:]

        E = self.E
        K = self.K


        # Scatter capacity into fixed edge order — this is the alignment step.
        capacity_by_edge = np.zeros(E, dtype=np.float32)
        for i, j, v in zip(cap_i, cap_j, cap_v):
            pos = self.edge_pos.get((int(i), int(j)))
            if pos is not None:
                capacity_by_edge[pos] = v

        # Same alignment for the flow target, per commodity.
        flow_by_edge = np.zeros((E, K), dtype=np.float32)
        for k, i, j, v in zip(flow_k, flow_i, flow_j, flow_v):
            pos = self.edge_pos.get((int(i), int(j)))
            if pos is not None:
                flow_by_edge[pos, k] = v

        distance_by_edge = self.static.edge_distance  # already in fixed order

        edge_attr = torch.tensor(
            np.stack([distance_by_edge, capacity_by_edge], axis=1), dtype=torch.float32
        )

        #We need to append 2N edge attributes with zeroes in distance and capacity

        cap_attr = torch.full((2*self.N,), 0.0, dtype=torch.float32)
        dist_attr = torch.full((2*self.N,), 0.0, dtype=torch.float32)
        edge_attr = torch.cat([edge_attr,torch.stack([dist_attr,cap_attr],dim=1)],dim=0)

        x = torch.tensor(
            np.concatenate([supply, demand, price], axis=1), dtype=torch.float32
            )

        #Adding the virtual node's placeholder row to X's output
        x = torch.cat([x, torch.zeros((1, 3*K), dtype=torch.float32)], dim=0)


        x = torch.log1p(x)
        edge_attr = torch.log1p(edge_attr)
        y = torch.log1p(torch.tensor(flow_by_edge, dtype=torch.float32))
        return Data(x=x, edge_index=self.edge_index, edge_attr=edge_attr,
                    edge_id=self.edge_id, y=y)






In [ ]:
#Building the custom message layer from PEW

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax
from torch_geometric.utils import add_self_loops
from torch import arange

class CustomRGATConv(MessagePassing):
    #
    def __init__(self, in_channels, out_channels,num_edges,d_attn,N,
                 edge_dim,heads=1, dropout=0.0):

        super().__init__(aggr='add', node_dim=0)

        #since we are self looping num_edges is the number of edges + cities

        self.in_channels = in_channels #size of incoming node feacture vectors
        self.out_channels = out_channels #size of output feature fectors
        self.heads = heads #Number of multihead attention mechanisms
        self.dropout = dropout
        self.N = N #Nmber of cities
        self.num_edges = num_edges #Num Edges is now cities+edges
        self.d_attn = d_attn
        self.edge_dim = edge_dim


        #defining the node embeddings from PEW

        self.msg_weight = nn.Embedding(num_edges,(in_channels) * heads * out_channels)
        self.query_weight = nn.Embedding(num_edges, (out_channels)* heads * d_attn)
        self.key_weight = nn.Embedding(num_edges,(out_channels)* heads *d_attn)

        #Bias vector
        #Bias vector for each edge

        self.bias_weight = nn.Embedding(num_edges,heads*out_channels)

        #shared linear projection matrix for shared attributes
        self.W = nn.Parameter(torch.Tensor(edge_dim,heads*d_attn))

        self.reset_parameters()

    def reset_parameters(self):


        #reset for msg weight
        fan_in = self.in_channels
        fan_out = self.heads * self.out_channels
        bound = (6.0 / (fan_in + fan_out)) ** 0.5
        nn.init.uniform_(self.msg_weight.weight, -bound, bound)


        #reset for query and key

        fan_in = self.out_channels
        fan_out = self.heads * self.d_attn
        bound = (6.0 / (fan_in + fan_out)) ** 0.5
        nn.init.uniform_(self.query_weight.weight, -bound, bound)
        nn.init.uniform_(self.key_weight.weight, -bound, bound)

        nn.init.xavier_uniform_(self.W)
        nn.init.zeros_(self.bias_weight.weight)


    def forward(self,x,edge_index,edge_id,edge_attr):

        #adding self-loops here
        #needed further downstream
        edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr, num_nodes=x.size(0), fill_value=0)

        #local_node_id = arange(x.size(0)) % self.N
        local_node_id = arange(x.size(0), device=x.device) % self.N
        self_loop_ids =  (self.num_edges - self.N) + local_node_id
        edge_id = torch.cat([edge_id, self_loop_ids], dim=0)


        out =  self.propagate(edge_index,x=x,edge_attr=edge_attr,
                              edge_id=edge_id, size = (x.size(0),x.size(0)))
        return out

    def message(self, x_i, x_j, edge_attr, index, edge_id, ptr, size_i):
        #Note h_j is source, h_i is destination
        #require reshape to proceed

        x_i = x_i.reshape(-1,1,self.in_channels)
        x_j = x_j.reshape(-1,1,self.in_channels)

        query = self.query_weight(edge_id)
        query = query.reshape(-1,self.out_channels,self.heads*self.d_attn)

        key = self.key_weight(edge_id)
        key = key.reshape(-1,self.out_channels,self.heads*self.d_attn)

        msg = self.msg_weight(edge_id)
        msg = msg.reshape(-1,self.in_channels,self.heads*self.out_channels)

        bias = self.bias_weight(edge_id)
        bias = bias.reshape(-1,1,self.heads*self.out_channels)


        #create the intermediate representation
        #multiply the node features with the per edge weight matrix

        #source
        g_s = torch.bmm(x_j,msg)
        g_s = g_s.reshape(-1,self.heads,self.out_channels)

        #destination
        g_d = torch.bmm(x_i,msg)
        g_d = g_d.reshape(-1,self.heads,self.out_channels)


        #query weight applied to destination
        q_e = torch.bmm(g_d,query)
        q_e = q_e.reshape(-1,self.heads,self.d_attn)
        #key weight applied to source
        k_e = torch.bmm(g_s,key)
        k_e = k_e.reshape(-1,self.heads,self.d_attn)


        #calculating the shared linear projection
        w = torch.matmul(edge_attr,self.W)
        w = w.reshape(-1,self.heads,self.d_attn)

        #Calculating the raw score
        raw_score = F.leaky_relu(q_e+k_e+w, negative_slope=0.01)

        #Reduce to one scalar score per edge per head
        #Note that PEW implementation doesn't specify
        #assuming plain sum could cahnge to a smalled learned reduction vector
        raw_score = raw_score.sum(dim=-1) / (self.d_attn ** .5)



        #closed neighborehood softmax
        soft_score = softmax(raw_score,index,ptr,num_nodes=size_i)
        soft_score = F.dropout(soft_score,p=self.dropout,training=self.training)

        update_value = g_s * soft_score.reshape(-1,self.heads,1) + bias
        return update_value


    def update(self, aggr_out):

        # Concatenate multi-head outputs back into a single vector

        return F.relu(aggr_out.view(-1, self.heads * self.out_channels))

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader



dataset = GraphTransportationDataset(
    shard_dir="/content/drive/MyDrive/training_data",
    static_network_path="/content/drive/MyDrive/training_data/static_network.h5"
)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

K = dataset.static.K
N = dataset.static.N
E = dataset.static.num_edges

In [ ]:
import glob, h5py, os

shards = sorted(glob.glob("/content/drive/MyDrive/training_data/shard_*.h5"))
print(len(shards), "shards")

bad, counts = [], {}
for p in shards:
    try:
        with h5py.File(p, "r") as f:
            has_attr = "num_simulations" in f.attrs
            n_groups = len(f["simulations"]) if "simulations" in f else 0
            counts[p] = (int(f.attrs["num_simulations"]) if has_attr else None, n_groups)
            if not has_attr:
                bad.append((os.path.basename(p), n_groups, os.path.getsize(p)))
    except Exception as e:
        bad.append((os.path.basename(p), "UNREADABLE", repr(e)))

print("missing attr / unreadable:", len(bad))
for b in bad: print(b)

mism = [(os.path.basename(p), a, g) for p, (a, g) in counts.items() if a is not None and a != g]
print("attr != group count:", mism)

200 shards
missing attr / unreadable: 1
('shard_00001.h5', 489, 9799736)
attr != group count: []


In [ ]:
D = "/content/drive/MyDrive/training_data"
with h5py.File(f"{D}/shard_00001.h5", "r+") as f:
    f.attrs["num_simulations"] = 489

In [ ]:

#Some deliberate items
#odel = CustomRGATConv(3*K,16,E+N,1,N,2,heads=1)

class PEWModel(nn.Module):
  def __init__(self, N, E, K, hidden_dims=(16, 32, 64), d_attn=16, edge_dim=2, heads=1):
    super().__init__()
    self.v = nn.Parameter(torch.zeros(1,3*K))
    self.ablate_virtual = False
    self.N = N
    self.E = E
    self.K = K
    self.hidden_dims = hidden_dims
    self.d_attn = d_attn
    self.edge_dim = edge_dim
    self.heads = heads

    self.conv1 = CustomRGATConv(3*K,hidden_dims[0],E+N+3,d_attn,N+1,edge_dim,heads=heads)
    self.conv2 = CustomRGATConv(hidden_dims[0],hidden_dims[1],E+N+3,d_attn,N+1,edge_dim,heads=heads)
    self.conv3 = CustomRGATConv(hidden_dims[1],hidden_dims[2],E+N+3,d_attn,N+1,edge_dim,heads=heads)
    self.projection_layer1= nn.Linear(hidden_dims[0],hidden_dims[1])
    self.projection_layer2= nn.Linear(hidden_dims[1],hidden_dims[2])
    self.edge_emb = nn.Embedding(E, 16)   # learned per-edge identity
    self.edge_head = nn.Sequential(
      nn.Linear(2 * hidden_dims[2] + edge_dim + 16, 64),
      nn.ReLU(),
      nn.Linear(64, K),
      )

    #layer norm
    self.norm1 = nn.LayerNorm(hidden_dims[0])
    self.norm2 = nn.LayerNorm(hidden_dims[1])
    self.norm3 = nn.LayerNorm(hidden_dims[2])

  def forward(self, x, edge_index, edge_id, edge_attr):
    B = x.size(0) // (self.N+1)
    #indices = torch.arange(B) * (self.N +1) + self.N
    indices = torch.arange(B, device=x.device) * (self.N + 1) + self.N
    x = x.clone()
    x[indices] = self.v
    h1 = self.norm1(self.conv1(x, edge_index, edge_id, edge_attr))
    if self.ablate_virtual:
      h1 = h1.clone()
      h1[indices] = h1[indices].mean(0,keepdim=True)
    h2 = self.norm2(self.conv2(h1, edge_index, edge_id, edge_attr) + self.projection_layer1(h1))
    if self.ablate_virtual:
      h2 = h2.clone()
      h2[indices] = h2[indices].mean(0,keepdim=True)
    h3 = self.norm3(self.conv3(h2, edge_index, edge_id, edge_attr) + self.projection_layer2(h2))

    #Drop the rows in
    edges_per_sample = self.E + 2 * self.N
    offsets = torch.arange(B, device=edge_index.device) * edges_per_sample          # [B]
    real_cols = (offsets.unsqueeze(1) + torch.arange(self.E, device=edge_index.device).unsqueeze(0)).reshape(-1)  # [B*E]

    edge_index = edge_index[:, real_cols]
    edge_attr = edge_attr[real_cols]

    src, dst = edge_index
    edge_feat = self.edge_emb(torch.arange(self.E, device=x.device).repeat(B))   # [B*E, 16]
    z = torch.cat([h3[src], h3[dst], edge_attr, edge_feat], dim=1)
    return F.softplus(self.edge_head(z)), edge_attr

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PEWModel(N=dataset.static.N, E=dataset.static.num_edges, K=dataset.static.K).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
ckpt = torch.load(os.path.join(checkpoint_dir, 'best_model_v5.pt'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

@torch.no_grad()
def test_report(loader):
    model.eval()
    sq_err = sq_base = n_el = 0.0
    abs_err = true_sum = 0.0
    edge_abs_err = edge_true_sum = 0.0
    n_edges = n_viol = 0

    for batch in loader:
        batch = batch.to(device)
        out, edge_attr = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
        y = batch.y

        # log space (same scale as training loss)
        sq_err  += ((out - y) ** 2).sum().item()
        sq_base += (y ** 2).sum().item()          # baseline: predict zero flow everywhere
        n_el    += y.numel()

        # real flow units
        pred, true = torch.expm1(out), torch.expm1(y)
        abs_err  += (pred - true).abs().sum().item()
        true_sum += true.sum().item()

        # total flow per edge (summed over commodities) and capacity violations
        pred_tot, true_tot = pred.sum(1), true.sum(1)
        edge_abs_err  += (pred_tot - true_tot).abs().sum().item()
        edge_true_sum += true_tot.sum().item()

        cap = torch.expm1(edge_attr[:, 1])
        n_viol  += (pred_tot > cap * 1.01).sum().item()
        n_edges += pred_tot.numel()

    print(f"Log-space MSE:                  {sq_err/n_el:.4f}   (predict-zero baseline: {sq_base/n_el:.4f})")
    print(f"Per-commodity flow error:       {100*abs_err/true_sum:.1f}% of total true flow")
    print(f"Per-edge total flow error:      {100*edge_abs_err/edge_true_sum:.1f}% of total true flow")
    print(f"Edges over capacity by >1%:     {100*n_viol/n_edges:.2f}%   (Pyomo's solutions: 0%)")

test_report(test_loader)

Loaded best checkpoint from epoch 30 (val_loss=8.0287)
Log-space MSE:                  4.7500   (predict-zero baseline: 6.6760)
Per-commodity flow error:       80.2% of total true flow
Per-edge total flow error:      76.8% of total true flow
Edges over capacity by >1%:     0.18%   (Pyomo's solutions: 0%)


In [ ]:
Y_train = torch.stack([d.y for d in train_data])            # [Ntr, E, K], log space
Y_test  = torch.stack([d.y for d in test_data])             # [Nte, E, K]
mean_pat = Y_train.mean(0, keepdim=True)

log_mse = F.mse_loss(mean_pat.expand_as(Y_test), Y_test).item()
real_err = (torch.expm1(mean_pat.expand_as(Y_test)) - torch.expm1(Y_test)).abs().sum() \
           / torch.expm1(Y_test).abs().sum()
print(f"mean-pattern log MSE {log_mse:.4f} | real-unit err {real_err.item()*100:.1f}%")

mean-pattern log MSE 5.1020 | real-unit err 100.0%


In [ ]:
diagnose(test_loader)

Entries with nonzero true flow:        7.6%
Total predicted / total true flow:     0.47
-- where true flow is zero --
  log-space MSE:                       4.2817
  entries predicted > 0.1 (log):       31.6%
  phantom flow as % of true total:     5.8%
-- where true flow is nonzero --
  log-space MSE:                       10.4732
  abs error as % of their true flow:   74.4%


In [ ]:
#Trying to diagnose the virtual node
import os, torch
from torch_geometric.loader import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# fresh model: same constructor arguments as the run being tested (v4 = default dims 16/32/64)
model = PEWModel(N=dataset.static.N,
                 E=dataset.static.num_edges,
                 K=dataset.static.K).to(device)

# load the weighted-run checkpoint from Drive
checkpoint_dir = "/content/drive/MyDrive/training_data/checkpoints"
ckpt = torch.load(os.path.join(checkpoint_dir, "best_model_v5.pt"), map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")
print("ablate_virtual =", model.ablate_virtual)      # should print False

# larger loader for the ablation (batch mean is closer to a constant)
big_loader = DataLoader(test_data, batch_size=64, shuffle=False)

Loaded epoch 30 (val_loss=8.0287)
ablate_virtual = False


In [ ]:
print("=== BASELINE (virtual node active) ===")
model.ablate_virtual = False
test_report(big_loader)
diagnose(big_loader)

print("\n=== ABLATED (virtual node neutralized) ===")
model.ablate_virtual = True
test_report(big_loader)
diagnose(big_loader)

model.ablate_virtual = False        # reset so later cells use the normal model

=== BASELINE (virtual node active) ===
Log-space MSE:                  4.7500   (predict-zero baseline: 6.6760)
Per-commodity flow error:       80.2% of total true flow
Per-edge total flow error:      76.8% of total true flow
Edges over capacity by >1%:     0.18%   (Pyomo's solutions: 0%)
Entries with nonzero true flow:        7.6%
Total predicted / total true flow:     0.47
-- where true flow is zero --
  log-space MSE:                       4.2817
  entries predicted > 0.1 (log):       31.6%
  phantom flow as % of true total:     5.8%
-- where true flow is nonzero --
  log-space MSE:                       10.4732
  abs error as % of their true flow:   74.4%

=== ABLATED (virtual node neutralized) ===
Log-space MSE:                  5.1262   (predict-zero baseline: 6.6760)
Per-commodity flow error:       81.7% of total true flow
Per-edge total flow error:      78.3% of total true flow
Edges over capacity by >1%:     0.21%   (Pyomo's solutions: 0%)
Entries with nonzero true flow:      

In [ ]:
E = dataset.static.num_edges
print(model.conv1.bias_weight.weight[E].abs().mean().item(),
      model.conv1.bias_weight.weight[:E].abs().mean().item())

0.031169043853878975 0.08649640530347824


In [ ]:
#Creating a deph check
class PEWModelDeep(nn.Module):
    def __init__(self, N, E, K, hidden_dims=(32, 64, 128, 128, 128, 128),
                 d_attn=32, edge_dim=2, heads=1):
        super().__init__()
        self.v = nn.Parameter(torch.zeros(1, 3*K))
        self.N, self.E, self.K = N, E, K
        self.ablate_virtual = False
        dims = [3*K] + list(hidden_dims)
        L = len(hidden_dims)
        self.convs = nn.ModuleList([
            CustomRGATConv(dims[i], dims[i+1], E+N+3, d_attn, N+1, edge_dim, heads=heads)
            for i in range(L)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for d in hidden_dims])
        # residual shortcuts for layers 1..L-1 (identity when the width doesn't change)
        self.proj = nn.ModuleList([
            nn.Linear(dims[i], dims[i+1]) if dims[i] != dims[i+1] else nn.Identity()
            for i in range(1, L)])
        self.edge_emb = nn.Embedding(E, 16)
        self.edge_head = nn.Sequential(
            nn.Linear(2*hidden_dims[-1] + edge_dim + 16, 64), nn.ReLU(), nn.Linear(64, K))

    def forward(self, x, edge_index, edge_id, edge_attr):
        B = x.size(0) // (self.N + 1)
        indices = torch.arange(B, device=x.device) * (self.N + 1) + self.N
        x = x.clone()
        x[indices] = self.v
        L = len(self.convs)

        h = self.norms[0](self.convs[0](x, edge_index, edge_id, edge_attr))
        for i in range(1, L):
            if self.ablate_virtual:
                h = h.clone(); h[indices] = h[indices].mean(0, keepdim=True)
            h = self.norms[i](self.convs[i](h, edge_index, edge_id, edge_attr) + self.proj[i-1](h))

        edges_per_sample = self.E + 2 * self.N
        offsets = torch.arange(B, device=edge_index.device) * edges_per_sample
        real_cols = (offsets.unsqueeze(1) + torch.arange(self.E, device=edge_index.device).unsqueeze(0)).reshape(-1)
        edge_index = edge_index[:, real_cols]
        edge_attr = edge_attr[real_cols]
        src, dst = edge_index
        edge_feat = self.edge_emb(torch.arange(self.E, device=x.device).repeat(B))
        z = torch.cat([h[src], h[dst], edge_attr, edge_feat], dim=1)
        return F.softplus(self.edge_head(z)), edge_attr

In [ ]:
ckpt = torch.load(os.path.join(checkpoint_dir, 'best_model_v8_64k.pt'), map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (val_loss={ckpt['val_loss']:.4f})")

@torch.no_grad()
def test_report(loader):
    model.eval()
    sq_err = sq_base = n_el = 0.0
    abs_err = true_sum = 0.0
    edge_abs_err = edge_true_sum = 0.0
    n_edges = n_viol = 0

    for batch in loader:
        batch = batch.to(device)
        out, edge_attr = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
        y = batch.y

        # log space (same scale as training loss)
        sq_err  += ((out - y) ** 2).sum().item()
        sq_base += (y ** 2).sum().item()          # baseline: predict zero flow everywhere
        n_el    += y.numel()

        # real flow units
        pred, true = torch.expm1(out), torch.expm1(y)
        abs_err  += (pred - true).abs().sum().item()
        true_sum += true.sum().item()

        # total flow per edge (summed over commodities) and capacity violations
        pred_tot, true_tot = pred.sum(1), true.sum(1)
        edge_abs_err  += (pred_tot - true_tot).abs().sum().item()
        edge_true_sum += true_tot.sum().item()

        cap = torch.expm1(edge_attr[:, 1])
        n_viol  += (pred_tot > cap * 1.01).sum().item()
        n_edges += pred_tot.numel()

    print(f"Log-space MSE:                  {sq_err/n_el:.4f}   (predict-zero baseline: {sq_base/n_el:.4f})")
    print(f"Per-commodity flow error:       {100*abs_err/true_sum:.1f}% of total true flow")
    print(f"Per-edge total flow error:      {100*edge_abs_err/edge_true_sum:.1f}% of total true flow")
    print(f"Edges over capacity by >1%:     {100*n_viol/n_edges:.2f}%   (Pyomo's solutions: 0%)")

test_report(test_loader)

Loaded best checkpoint from epoch 28 (val_loss=8.8436)
Log-space MSE:                  4.5045   (predict-zero baseline: 6.7309)
Per-commodity flow error:       70.3% of total true flow
Per-edge total flow error:      66.6% of total true flow
Edges over capacity by >1%:     0.07%   (Pyomo's solutions: 0%)


In [ ]:
diagnose(test_loader)

Entries with nonzero true flow:        7.6%
Total predicted / total true flow:     0.60
-- where true flow is zero --
  log-space MSE:                       4.0656
  entries predicted > 0.1 (log):       30.0%
  phantom flow as % of true total:     6.9%
-- where true flow is nonzero --
  log-space MSE:                       9.8376
  abs error as % of their true flow:   63.4%


In [ ]:
@torch.no_grad()
def lp_metrics(loader, rate=0.05):
    model.eval()
    N, E, K = dataset.N, dataset.E, dataset.K
    src = dataset.edge_index[0, :E].to(device)     # tail i of each real arc
    dst = dataset.edge_index[1, :E].to(device)     # head j

    def score(flow, supply, demand, arc_cost, pen):          # flow: [B, E, K]
        B = flow.size(0)
        inflow  = torch.zeros(B, N, K, device=flow.device).index_add_(1, dst, flow)
        outflow = torch.zeros(B, N, K, device=flow.device).index_add_(1, src, flow)
        r = inflow - outflow - (supply - demand)
        unmet, unused = r.clamp(min=0), (-r).clamp(min=0)
        obj = (arc_cost * flow).sum((1, 2)) + pen * unmet.sum((1, 2))
        return obj, unmet.sum((1, 2)), unused.sum((1, 2)), outflow - inflow

    S = {k: [] for k in ["obj_p", "obj_t", "unm_p", "unm_t", "unu_p", "unu_t", "dem"]}
    net_at_supply = []
    for batch in loader:
        batch = batch.to(device)
        B = batch.num_graphs
        out, edge_attr = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
        pred = torch.expm1(out).view(B, E, K)
        true = torch.expm1(batch.y).view(B, E, K)
        x = torch.expm1(batch.x).view(B, N + 1, 3 * K)[:, :N]          # drop the virtual node
        supply, demand, price = x[..., :K], x[..., K:2*K], x[..., 2*K:]
        dist = torch.expm1(edge_attr[:, 0]).view(B, E, 1)
        arc_cost = price[:, src, :] + rate * dist                       # [B, E, K]
        pen = 5 * arc_cost.amax(dim=(1, 2))                             # [B]

        op, up, nup, _      = score(pred, supply, demand, arc_cost, pen)
        ot, ut, nut, netout = score(true, supply, demand, arc_cost, pen)
        for k, v in zip(S, [op, ot, up, ut, nup, nut, demand.sum((1, 2))]):
            S[k].append(v)
        net_at_supply.append(netout[(supply - demand) > 0])

    S = {k: torch.cat(v) for k, v in S.items()}
    print(f"Objective:      LP {S['obj_t'].mean():.4g} | model {S['obj_p'].mean():.4g} "
          f"| ratio {S['obj_p'].sum() / S['obj_t'].sum():.3f}")
    print(f"Unmet demand:   LP {100*S['unm_t'].sum()/S['dem'].sum():.1f}% of demand "
          f"| model {100*S['unm_p'].sum()/S['dem'].sum():.1f}%")
    print(f"Unused supply:  LP {S['unu_t'].mean():.4g} | model {S['unu_p'].mean():.4g}")
    print(f"Sign check: mean (outflow - inflow) at supply nodes, true flows = "
          f"{torch.cat(net_at_supply).mean():.4g}  (negative => supply nodes absorb)")


In [ ]:
#New run on fixed data
dataset = GraphTransportationDataset(
    shard_dir="/content/drive/MyDrive/training_data",
    static_network_path="/content/drive/MyDrive/training_data/static_network.h5",
)
n = len(dataset)
print("samples:", n)                     # expect 101989

VAL_SIZE, TEST_SIZE = 10000, 10000
TRAIN_SIZE = n - VAL_SIZE - TEST_SIZE    # 81,989

g = torch.Generator().manual_seed(42)
perm = torch.randperm(n, generator=g).tolist()
train_idx = perm[:TRAIN_SIZE]
val_idx   = perm[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
test_idx  = perm[TRAIN_SIZE + VAL_SIZE:]

samples: 101989


In [ ]:
from torch.utils.data import Subset
from torch_geometric.loader import DataLoader

BATCH_SIZE = 8   # same as before

train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))   # 81989 10000 10000

81989 10000 10000


In [ ]:
import torch

@torch.no_grad()
def lp_metrics(loader, predict_fn, dataset, rate=0.05, device="cuda", max_batches=None):
    """
    predict_fn(batch) -> predicted flows in REAL units, shape (B*E, K).
      - truth check:  predict_fn = lambda b: b.y
      - model:        see the example below
    Assumes batch.x / batch.edge_attr / batch.y are the raw tensors that the
    dataset class returns (before any log1p).
    """
    N, E, K = dataset.N, dataset.E, dataset.K
    tail = torch.as_tensor(dataset.static.edge_i, dtype=torch.long, device=device)
    head = torch.as_tensor(dataset.static.edge_j, dtype=torch.long, device=device)

    def objective(flow, price, dist, supply, demand):
        """flow: (B,E,K). Returns obj, unmet, unused, max_arc_cost."""
        B = flow.size(0)
        arc_cost = price[:, tail, :] + rate * dist.unsqueeze(-1)        # (B,E,K)
        max_arc = arc_cost.amax(dim=(1, 2))                              # (B,)
        out = torch.zeros(B, N, K, device=flow.device).index_add_(1, tail, flow)
        inn = torch.zeros(B, N, K, device=flow.device).index_add_(1, head, flow)
        r = out - inn - (supply - demand)       # = unmet - unused at the optimum
        unmet, unused = torch.relu(r), torch.relu(-r)
        obj = (arc_cost * flow).sum(dim=(1, 2)) + 5.0 * max_arc * unmet.sum(dim=(1, 2))
        return obj, unmet, unused, out, inn

    tot = dict(pred_obj=0., lp_obj=0., unmet=0., lp_unmet=0., demand=0.,
               unused=0., cap_viol=0., cap_total=0., edges_viol=0., edges=0,
               surplus_net=0., surplus_n=0, samples=0)
    ratios = []

    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        batch = batch.to(device)
        B = batch.num_graphs

        x = torch.expm1(batch.x.view(B, N + 1, 3 * K)[:, :N])             # drop virtual node row
        supply, demand, price = x[..., :K], x[..., K:2 * K], x[..., 2 * K:]
        ea = torch.expm1(batch.edge_attr.view(B, E + 2 * N, 2)[:, :E])       # drop virtual/self rows
        dist, cap = ea[..., 0], ea[..., 1]

        y_true = torch.expm1(batch.y.view(B, E, K))
        y_pred = predict_fn(batch).view(B, E, K).clamp_min(0)

        obj_p, unmet_p, unused_p, out_p, inn_p = objective(y_pred, price, dist, supply, demand)
        obj_t, unmet_t, _, _, _ = objective(y_true, price, dist, supply, demand)

        viol = torch.relu(y_pred.sum(-1) - cap)                  # (B,E)

        surplus = supply > demand
        if surplus.any():
            tot["surplus_net"] += (out_p - inn_p)[surplus].sum().item()
            tot["surplus_n"] += surplus.sum().item()

        tot["pred_obj"] += obj_p.sum().item();  tot["lp_obj"] += obj_t.sum().item()
        tot["unmet"] += unmet_p.sum().item();   tot["lp_unmet"] += unmet_t.sum().item()
        tot["demand"] += demand.sum().item();   tot["unused"] += unused_p.sum().item()
        tot["cap_viol"] += viol.sum().item();   tot["cap_total"] += cap.sum().item()
        tot["edges_viol"] += (viol > 1e-3 * cap.clamp_min(1)).sum().item()
        tot["edges"] += B * E;                  tot["samples"] += B
        ratios.append((obj_p / obj_t.clamp_min(1e-6)).cpu())

    ratios = torch.cat(ratios)
    res = {
        "objective ratio (sum pred / sum LP)": tot["pred_obj"] / tot["lp_obj"],
        "objective ratio per-sample median":   ratios.median().item(),
        "objective ratio per-sample 90th pct": ratios.quantile(0.9).item(),
        "unmet demand % (model)":              100 * tot["unmet"] / tot["demand"],
        "unmet demand % (LP)":                 100 * tot["lp_unmet"] / tot["demand"],
        "unused supply per sample (model)":    tot["unused"] / tot["samples"],
        "capacity violation (% of capacity)":  100 * tot["cap_viol"] / tot["cap_total"],
        "edges over capacity %":               100 * tot["edges_viol"] / tot["edges"],
        "mean net outflow at surplus nodes":   tot["surplus_net"] / max(tot["surplus_n"], 1),
    }
    for k, v in res.items():
        print(f"{k:40s} {v:12.4f}")
    return res

In [ ]:
lp_metrics(val_loader, lambda b: b.y, dataset, max_batches=20)

objective ratio (sum pred / sum LP)            5.5429
objective ratio per-sample median             11.9876
objective ratio per-sample 90th pct           26.7094
unmet demand % (model)                        99.9967
unmet demand % (LP)                           13.9041
unused supply per sample (model)         3419082.9125
capacity violation (% of capacity)             0.0000
edges over capacity %                          0.0000
mean net outflow at surplus nodes              8.6276


{'objective ratio (sum pred / sum LP)': 5.542858539718671,
 'objective ratio per-sample median': 11.987629890441895,
 'objective ratio per-sample 90th pct': 26.709409713745117,
 'unmet demand % (model)': 99.99672104179493,
 'unmet demand % (LP)': 13.904087534608985,
 'unused supply per sample (model)': 3419082.9125,
 'capacity violation (% of capacity)': 0.0,
 'edges over capacity %': 0.0,
 'mean net outflow at surplus nodes': 8.627581888834635}

In [ ]:
lp_metrics(val_loader, lambda b: torch.expm1(b.y), dataset, max_batches=20)

objective ratio (sum pred / sum LP)            1.0000
objective ratio per-sample median              1.0000
objective ratio per-sample 90th pct            1.0000
unmet demand % (model)                        13.9041
unmet demand % (LP)                           13.9041
unused supply per sample (model)          869616.2656
capacity violation (% of capacity)             0.0000
edges over capacity %                          0.0000
mean net outflow at surplus nodes         169970.9629


{'objective ratio (sum pred / sum LP)': 1.0,
 'objective ratio per-sample median': 1.0,
 'objective ratio per-sample 90th pct': 1.0,
 'unmet demand % (model)': 13.904087534608985,
 'unmet demand % (LP)': 13.904087534608985,
 'unused supply per sample (model)': 869616.265625,
 'capacity violation (% of capacity)': 8.702767422063182e-09,
 'edges over capacity %': 0.0,
 'mean net outflow at surplus nodes': 169970.96291666667}

In [30]:
import os, glob, shutil, time
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

#  Copy shards from Drive to local Colab disk (always fresh)
local_dir = "/content/training_data"
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
shutil.copytree("/content/drive/MyDrive/training_data", local_dir)
print(len(glob.glob(f"{local_dir}/shard_*.h5")), "shards")
!du -sh /content/training_data && df -h /content | tail -1

# Dataset from the local copy
dataset = GraphTransportationDataset(
    shard_dir=local_dir,
    static_network_path=os.path.join(local_dir, "static_network.h5"),
)
N, E, K = dataset.N, dataset.E, dataset.K
print("samples:", len(dataset), "| N,E,K:", N, E, K)

# Split (random permutation, seed 42) and preload into RAM
torch.manual_seed(42)
VAL_SIZE, TEST_SIZE = 10000, 10000
TRAIN_SIZE = len(dataset) - VAL_SIZE - TEST_SIZE

perm = torch.randperm(len(dataset))
train_idx = perm[:TRAIN_SIZE]
val_idx   = perm[TRAIN_SIZE : TRAIN_SIZE + VAL_SIZE]
test_idx  = perm[TRAIN_SIZE + VAL_SIZE :]

t = time.time()
train_data = [dataset[i] for i in train_idx.tolist()]
print(f"train preload: {time.time()-t:.1f}s")
t = time.time()
val_data = [dataset[i] for i in val_idx.tolist()]
print(f"val preload: {time.time()-t:.1f}s")
test_data = [dataset[i] for i in test_idx.tolist()]

train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=8, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=8, shuffle=False)
print(len(train_data), len(val_data), len(test_data))

# Loss functions (all inputs/outputs in log1p space unless stated)
def capcity_penalty(out, edge_attr, eps=1e-6):
    out = torch.expm1(out)
    out = out.sum(dim=1)
    edge_attr = torch.expm1(edge_attr[:, 1])
    return (F.relu((out - edge_attr) / (edge_attr + eps)) ** 2).mean()

def weighted_mse(out, y, w_nz=5.0):
    w = 1.0 + (w_nz - 1.0) * (y > 0).float()      # y>0 <=> flow>0 in log1p space
    return (w * (out - y) ** 2).mean()

def real_unit_loss(out, y, edge_attr, eps=1e-6):
    cap  = torch.expm1(edge_attr[:, 1]).unsqueeze(1) + eps
    pred = torch.expm1(out.clamp(max=20))
    true = torch.expm1(y)
    num = (((pred - true) / cap) ** 2).mean()
    den = ((true / cap) ** 2).mean() + eps
    return num / den                               # 1.0 = predict-zero level, 0 = perfect

tail = torch.as_tensor(dataset.static.edge_i, dtype=torch.long, device=device)
head = torch.as_tensor(dataset.static.edge_j, dtype=torch.long, device=device)

def objective_loss(out, y, batch, edge_attr, rate=0.05):
    """Predicted LP objective / true LP objective, mean over samples (1.0 = optimal)."""
    B = batch.num_graphs
    pred = torch.expm1(out.clamp(max=20)).view(B, E, K)
    true = torch.expm1(y).view(B, E, K)
    x = torch.expm1(batch.x.view(B, N + 1, 3 * K)[:, :N])
    supply, demand, price = x[..., :K], x[..., K:2*K], x[..., 2*K:]
    dist = torch.expm1(edge_attr[:, 0]).view(B, E)

    arc_cost = price[:, tail, :] + rate * dist.unsqueeze(-1)          # (B,E,K)
    max_arc = arc_cost.amax(dim=(1, 2))                                # (B,)

    def obj(flow):
        outf = torch.zeros(B, N, K, device=flow.device).index_add_(1, tail, flow)
        inf_ = torch.zeros(B, N, K, device=flow.device).index_add_(1, head, flow)
        unmet = torch.relu(outf - inf_ - (supply - demand))
        return (arc_cost * flow).sum(dim=(1, 2)) + 5.0 * max_arc * unmet.sum(dim=(1, 2))

    return (obj(pred) / obj(true).clamp_min(1.0)).mean()

# ---- 5. Training / evaluation ----
def _step(batch, lamb, lam_real, lam_obj):
    out, edge_attr = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
    penalty   = capcity_penalty(out, edge_attr)
    data_loss = weighted_mse(out, batch.y)
    real_loss = real_unit_loss(out, batch.y, edge_attr)
    obj_loss  = (objective_loss(out, batch.y, batch, edge_attr)
                 if lam_obj > 0 else out.new_zeros(()))
    loss = data_loss + lamb * penalty + lam_real * real_loss + lam_obj * obj_loss
    return loss, data_loss, penalty, real_loss, obj_loss

def _epoch(loader, train, lamb, lam_real, lam_obj):
    model.train(train)
    acc = [0.0] * 5
    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(device)
            if train:
                optimizer.zero_grad()
            terms = _step(batch, lamb, lam_real, lam_obj)
            if train:
                terms[0].backward()
                optimizer.step()
            for i, v in enumerate(terms):
                acc[i] += v.item()
    return [a / len(loader) for a in acc]

def train_one_epoch(loader, lamb=1.0, lam_real=3.0, lam_obj=0.0):
    return _epoch(loader, True, lamb, lam_real, lam_obj)

def evaluate(loader, lamb=1.0, lam_real=3.0, lam_obj=0.0):
    return _epoch(loader, False, lamb, lam_real, lam_obj)

def run(lam_obj, tag, epochs=30, loader=None):
    global model, optimizer
    loader = loader or train_loader
    model = PEWModel(N, E, K, hidden_dims=(32, 64, 128), d_attn=32).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best = float("inf")
    for ep in range(1, epochs + 1):
        tr = train_one_epoch(loader, lam_obj=lam_obj)
        va = evaluate(val_loader, lam_obj=1.0)
        sched.step()
        if va[4] < best:
            best = va[4]
            torch.save(model.state_dict(), f"/content/drive/MyDrive/best_{tag}.pt")   # survives a disconnect
        print(f"ep {ep:2d} | train {tr[0]:.4f} | val real {va[3]:.4f} | val obj ratio {va[4]:.4f}")
    return best

small_loader = DataLoader(train_data[:40000], batch_size=8, shuffle=True)

run(0.0, "small_base", epochs=15, loader=small_loader)
run(1.0, "small_obj",  epochs=15, loader=small_loader)

200 shards
2.0G	/content/training_data
overlay         113G   49G   65G  43% /
samples: 101989 | N,E,K: 110 240 5
train preload: 358.6s
val preload: 44.8s
81989 10000 10000
ep  1 | train 13.1590 | val real 0.7675 | val obj ratio 13.8920
ep  2 | train 11.5936 | val real 0.6888 | val obj ratio 15.1718
ep  3 | train 11.0119 | val real 0.6486 | val obj ratio 14.5663
ep  4 | train 10.5244 | val real 0.5946 | val obj ratio 15.5785
ep  5 | train 10.1976 | val real 0.5672 | val obj ratio 12.7405
ep  6 | train 9.9530 | val real 0.5521 | val obj ratio 11.8524
ep  7 | train 9.7469 | val real 0.5219 | val obj ratio 12.1413
ep  8 | train 9.5803 | val real 0.5346 | val obj ratio 10.7745
ep  9 | train 9.4496 | val real 0.5072 | val obj ratio 10.7598
ep 10 | train 9.3320 | val real 0.4999 | val obj ratio 11.2586
ep 11 | train 9.2355 | val real 0.4928 | val obj ratio 10.8763
ep 12 | train 9.1542 | val real 0.4934 | val obj ratio 10.5758
ep 13 | train 9.0892 | val real 0.4883 | val obj ratio 10.4911
ep 

6.55296338262558

In [31]:
def eval_ckpt(tag):
    global model
    model = PEWModel(N, E, K, hidden_dims=(32, 64, 128), d_attn=32).to(device)
    model.load_state_dict(torch.load(f"/content/drive/MyDrive/best_{tag}.pt"))
    model.eval()
    print(f"=== {tag} ===")
    return lp_metrics(test_loader, model_predict, dataset)

eval_ckpt("small_base")
eval_ckpt("small_obj")

=== small_base ===
objective ratio (sum pred / sum LP)              4.1805
objective ratio per-sample median                7.4308
objective ratio per-sample 90th pct             21.9025
unmet demand % (model)                          84.4064
unmet demand % (LP)                             17.1351
unused supply per sample (model)           2968680.1796
capacity violation (% of capacity)               0.0006
edges over capacity %                            0.0818
mean net outflow at surplus nodes           146582.3313
=== small_obj ===
objective ratio (sum pred / sum LP)              2.9054
objective ratio per-sample median                4.8150
objective ratio per-sample 90th pct             13.5985
unmet demand % (model)                          57.5955
unmet demand % (LP)                             17.1351
unused supply per sample (model)           2192341.5108
capacity violation (% of capacity)               0.0124
edges over capacity %                            0.2983
mean net ou

{'objective ratio (sum pred / sum LP)': 2.9053523319931522,
 'objective ratio per-sample median': 4.814958095550537,
 'objective ratio per-sample 90th pct': 13.598474502563477,
 'unmet demand % (model)': 57.59552767196305,
 'unmet demand % (LP)': 17.135115802049516,
 'unused supply per sample (model)': 2192341.5108,
 'capacity violation (% of capacity)': 0.012442508497290814,
 'edges over capacity %': 0.2982916666666667,
 'mean net outflow at surplus nodes': 151910.5516}

In [ ]:
@torch.no_grad()
def lp_metrics(loader, predict_fn, dataset, rate=0.05, max_batches=None):
    """predict_fn(batch) -> predicted flows in REAL units (B*E, K).
       Batches hold log1p-scaled x / edge_attr / y; they are undone here."""
    tl = torch.as_tensor(dataset.static.edge_i, dtype=torch.long, device=device)
    hd = torch.as_tensor(dataset.static.edge_j, dtype=torch.long, device=device)

    def objective(flow, price, dist, supply, demand):
        B = flow.size(0)
        arc_cost = price[:, tl, :] + rate * dist.unsqueeze(-1)
        max_arc = arc_cost.amax(dim=(1, 2))
        out = torch.zeros(B, N, K, device=flow.device).index_add_(1, tl, flow)
        inn = torch.zeros(B, N, K, device=flow.device).index_add_(1, hd, flow)
        r = out - inn - (supply - demand)
        unmet, unused = torch.relu(r), torch.relu(-r)
        obj = (arc_cost * flow).sum(dim=(1, 2)) + 5.0 * max_arc * unmet.sum(dim=(1, 2))
        return obj, unmet, unused, out, inn

    tot = dict(pred_obj=0., lp_obj=0., unmet=0., lp_unmet=0., demand=0., unused=0.,
               cap_viol=0., cap_total=0., edges_viol=0, edges=0,
               surplus_net=0., surplus_n=0, samples=0)
    ratios = []

    for bi, batch in enumerate(loader):
        if max_batches is not None and bi >= max_batches:
            break
        batch = batch.to(device)
        B = batch.num_graphs

        x = torch.expm1(batch.x.view(B, N + 1, 3 * K)[:, :N])
        supply, demand, price = x[..., :K], x[..., K:2*K], x[..., 2*K:]
        ea = torch.expm1(batch.edge_attr.view(B, E + 2 * N, 2)[:, :E])
        dist, cap = ea[..., 0], ea[..., 1]

        y_true = torch.expm1(batch.y.view(B, E, K))
        y_pred = predict_fn(batch).view(B, E, K).clamp_min(0)

        obj_p, unmet_p, unused_p, out_p, inn_p = objective(y_pred, price, dist, supply, demand)
        obj_t, unmet_t, _, _, _ = objective(y_true, price, dist, supply, demand)
        viol = torch.relu(y_pred.sum(-1) - cap)

        surplus = supply > demand
        if surplus.any():
            tot["surplus_net"] += (out_p - inn_p)[surplus].sum().item()
            tot["surplus_n"] += surplus.sum().item()

        tot["pred_obj"] += obj_p.sum().item();   tot["lp_obj"] += obj_t.sum().item()
        tot["unmet"] += unmet_p.sum().item();    tot["lp_unmet"] += unmet_t.sum().item()
        tot["demand"] += demand.sum().item();    tot["unused"] += unused_p.sum().item()
        tot["cap_viol"] += viol.sum().item();    tot["cap_total"] += cap.sum().item()
        tot["edges_viol"] += (viol > 1e-3 * cap.clamp_min(1)).sum().item()
        tot["edges"] += B * E;                   tot["samples"] += B
        ratios.append((obj_p / obj_t.clamp_min(1e-6)).cpu())

    ratios = torch.cat(ratios)
    res = {
        "objective ratio (sum pred / sum LP)":  tot["pred_obj"] / tot["lp_obj"],
        "objective ratio per-sample median":    ratios.median().item(),
        "objective ratio per-sample 90th pct":  ratios.quantile(0.9).item(),
        "unmet demand % (model)":               100 * tot["unmet"] / tot["demand"],
        "unmet demand % (LP)":                  100 * tot["lp_unmet"] / tot["demand"],
        "unused supply per sample (model)":     tot["unused"] / tot["samples"],
        "capacity violation (% of capacity)":   100 * tot["cap_viol"] / tot["cap_total"],
        "edges over capacity %":                100 * tot["edges_viol"] / tot["edges"],
        "mean net outflow at surplus nodes":    tot["surplus_net"] / max(tot["surplus_n"], 1),
    }
    for k, v in res.items():
        print(f"{k:40s} {v:14.4f}")
    return res

def model_predict(batch):
    out, _ = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
    return torch.expm1(out.clamp(max=20))

# ---- checks before training ----
# (1) truth vs truth: ratio 1.0, unmet equal to LP, zero capacity violation, net outflow ~1e5
lp_metrics(val_loader, lambda b: torch.expm1(b.y), dataset, max_batches=20)

# (2) objective_loss on true flows must be exactly 1.0
batch = next(iter(val_loader)).to(device)
B = batch.num_graphs
ea_edges = batch.edge_attr.view(B, E + 2 * N, 2)[:, :E].reshape(-1, 2)
print("objective_loss(truth):", objective_loss(batch.y, batch.y, batch, ea_edges).item())  # 1.0

# (3) shapes of the model outputs
model = PEWModel(N, E, K, hidden_dims=(32, 64, 128), d_attn=32).to(device)
out, ea = model(batch.x, batch.edge_index, batch.edge_id, batch.edge_attr)
print(out.shape, ea.shape)        # expect (B*E, K) and (B*E, 2)

objective ratio (sum pred / sum LP)              1.0000
objective ratio per-sample median                1.0000
objective ratio per-sample 90th pct              1.0000
unmet demand % (model)                          13.9041
unmet demand % (LP)                             13.9041
unused supply per sample (model)            869616.2656
capacity violation (% of capacity)               0.0000
edges over capacity %                            0.0000
mean net outflow at surplus nodes           169970.9629
objective_loss(truth): 1.0
torch.Size([1920, 5]) torch.Size([1920, 2])
